In [1]:
import os
import json
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

from PIL import Image
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

In [2]:
SPLITS = "/kaggle/input/datasets/iwmm10/chestxray-capstone-splits"
BASE   = f"{SPLITS}/capstone_splits"
NIH    = "/kaggle/input/datasets/nih-chest-xrays/data"
OUT    = "/kaggle/working"

SPLIT_DIR  = f"{BASE}/splits"
INDEX_JSON = f"{SPLITS}/image_index.json"

IMG_COL, PID_COL = "Image Index", "Patient ID"

LABELS = [
    "Atelectasis", "Consolidation", "Infiltration", "Pneumothorax",
    "Edema", "Emphysema", "Fibrosis", "Effusion",
    "Pneumonia", "Pleural_Thickening", "Cardiomegaly", "Nodule",
    "Mass", "Hernia",
]

IMG_SIZE, SEED = 224, 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open(INDEX_JSON) as f:
    index = json.load(f)

def image_path(fn):
    return f"{NIH}/{index[fn]}/images/{fn}"

print("device:", device, "|", f"{len(index):,} images indexed")

device: cuda | 112,120 images indexed


In [3]:
eval_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class ChestXrayDataset(Dataset):
    def __init__(self, df, transform):
        self.files  = df[IMG_COL].values
        self.labels = df[LABELS].values.astype("float32")
        self.transform = transform
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        img = Image.open(image_path(self.files[i])).convert("RGB")
        return self.transform(img), torch.from_numpy(self.labels[i])

val_df  = pd.read_csv(f"{SPLIT_DIR}/val.csv")
test_df = pd.read_csv(f"{SPLIT_DIR}/test.csv")

def make_loader(df):
    return DataLoader(ChestXrayDataset(df, eval_tf), batch_size=32,
                      shuffle=False, num_workers=4, pin_memory=True)

val_loader, test_loader = make_loader(val_df), make_loader(test_df)
print(f"val {len(val_df):,} | test {len(test_df):,}")

val 3,978 | test 3,904


In [4]:
def predict(checkpoint_path, arch, loader):
    """Run any checkpoint over a loader. arch: 'densenet' or 'resnet'."""
    if arch == "densenet":
        m = models.densenet121(weights=None)
        m.classifier = nn.Linear(m.classifier.in_features, len(LABELS))
    elif arch == "resnet":
        m = models.resnet50(weights=None)
        m.fc = nn.Linear(m.fc.in_features, len(LABELS))
    else:
        raise ValueError(f"unknown arch: {arch}")

    sd = torch.load(checkpoint_path, map_location="cpu")

    # Project standard is 14 outputs. 'No Finding' is the absence of the 14,
    # not a class — including it inflates macro AUROC and breaks Grad-CAM.
    out_dim = list(sd.values())[-1].shape[0]
    if out_dim != len(LABELS):
        raise ValueError(
            f"Checkpoint has {out_dim} outputs; project standard is {len(LABELS)}. "
            f"Slice the output layer or recompute over the 14 pathology columns."
        )

    m.load_state_dict(sd)
    m.to(device).eval()

    probs, targets = [], []
    with torch.no_grad():
        for xb, yb in loader:
            with torch.amp.autocast('cuda'):
                out = m(xb.to(device, non_blocking=True))
            probs.append(torch.sigmoid(out).float().cpu())
            targets.append(yb)

    return torch.cat(probs).numpy(), torch.cat(targets).numpy()

In [5]:
def report(p, t, name):
    print(f"\n{'='*52}\n{name}\n{'='*52}")
    print(f"{'label':<20}{'AUROC':>9}{'AP':>9}{'n_pos':>8}")

    rows, aucs = [], []
    for i, label in enumerate(LABELS):
        if t[:, i].sum() == 0:
            continue
        auc = roc_auc_score(t[:, i], p[:, i])
        ap  = average_precision_score(t[:, i], p[:, i])
        aucs.append(auc)
        rows.append((label, auc, ap, int(t[:, i].sum())))
        print(f"{label:<20}{auc:>9.4f}{ap:>9.4f}{int(t[:, i].sum()):>8,}")

    macro = float(np.mean(aucs))
    print(f"{'-'*46}\n{'MACRO AUROC':<20}{macro:>9.4f}")
    return pd.DataFrame(rows, columns=["label", "auroc", "ap", "n_pos"]), macro

In [6]:
CKPT = "/kaggle/input/datasets/iwmm10/baseline/baseline_best (1).pt"

p_test, t_test = predict(CKPT, "densenet", test_loader)
df_dense, macro_dense = report(p_test, t_test, "DenseNet-121 — TEST")


DenseNet-121 — TEST
label                   AUROC       AP   n_pos
Atelectasis            0.7576   0.3927     612
Consolidation          0.7407   0.2132     333
Infiltration           0.6887   0.3550     838
Pneumothorax           0.8263   0.3451     375
Edema                  0.9049   0.4881     310
Emphysema              0.8890   0.5890     380
Fibrosis               0.7903   0.2454     212
Effusion               0.8296   0.5686     770
Pneumonia              0.6784   0.0964     163
Pleural_Thickening     0.7513   0.2523     309
Cardiomegaly           0.8869   0.5256     288
Nodule                 0.7366   0.2865     386
Mass                   0.7858   0.3496     374
Hernia                 0.8608   0.2894      30
----------------------------------------------
MACRO AUROC            0.7948


In [7]:
def bootstrap_ci(y_true, y_prob, n=1000, seed=42):
    rng = np.random.default_rng(seed)
    scores = []
    for _ in range(n):
        idx = rng.integers(0, len(y_true), len(y_true))
        if y_true[idx].sum() == 0:
            continue
        scores.append(roc_auc_score(y_true[idx], y_prob[idx]))
    return np.percentile(scores, [2.5, 97.5])

print(f"{'label':<20}{'AUROC':>9}{'95% CI':>20}")
for i, label in enumerate(LABELS):
    auc = roc_auc_score(t_test[:, i], p_test[:, i])
    lo, hi = bootstrap_ci(t_test[:, i], p_test[:, i])
    width = hi - lo
    flag = "  <-- wide" if width > 0.10 else ""
    print(f"{label:<20}{auc:>9.4f}   [{lo:.3f}, {hi:.3f}]{flag}")

label                   AUROC              95% CI
Atelectasis            0.7576   [0.736, 0.778]
Consolidation          0.7407   [0.713, 0.767]
Infiltration           0.6887   [0.668, 0.708]
Pneumothorax           0.8263   [0.803, 0.846]
Edema                  0.9049   [0.888, 0.921]
Emphysema              0.8890   [0.869, 0.908]
Fibrosis               0.7903   [0.763, 0.821]
Effusion               0.8296   [0.814, 0.846]
Pneumonia              0.6784   [0.633, 0.720]
Pleural_Thickening     0.7513   [0.723, 0.781]
Cardiomegaly           0.8869   [0.864, 0.906]
Nodule                 0.7366   [0.709, 0.762]
Mass                   0.7858   [0.760, 0.811]
Hernia                 0.8608   [0.790, 0.932]  <-- wide


In [8]:
p_val, t_val = predict(CKPT, "densenet", val_loader)

# Pick each class threshold on validation only, never on test
grid = np.arange(0.05, 0.95, 0.01)
thresholds = {}
for i, label in enumerate(LABELS):
    f1s = [f1_score(t_val[:, i], (p_val[:, i] >= th).astype(int), zero_division=0)
           for th in grid]
    thresholds[label] = float(grid[int(np.argmax(f1s))])

print(f"{'label':<20}{'thresh':>8}{'F1@0.5':>9}{'F1@tuned':>10}")
for i, label in enumerate(LABELS):
    f1_default = f1_score(t_test[:, i], (p_test[:, i] >= 0.5).astype(int), zero_division=0)
    f1_tuned   = f1_score(t_test[:, i], (p_test[:, i] >= thresholds[label]).astype(int), zero_division=0)
    print(f"{label:<20}{thresholds[label]:>8.2f}{f1_default:>9.3f}{f1_tuned:>10.3f}")

label                 thresh   F1@0.5  F1@tuned
Atelectasis             0.20    0.302     0.418
Consolidation           0.14    0.000     0.302
Infiltration            0.29    0.230     0.443
Pneumothorax            0.28    0.326     0.427
Edema                   0.26    0.368     0.518
Emphysema               0.36    0.521     0.574
Fibrosis                0.11    0.037     0.260
Effusion                0.24    0.457     0.565
Pneumonia               0.07    0.000     0.157
Pleural_Thickening      0.14    0.025     0.304
Cardiomegaly            0.23    0.482     0.537
Nodule                  0.14    0.128     0.319
Mass                    0.28    0.295     0.353
Hernia                  0.09    0.000     0.333


In [9]:
import json
print(open(f"{OUT}/thresholds.json").read())

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/thresholds.json'